In [3]:
import os
import cv2
import numpy as np
from glob import glob
from shutil import copy2
import random

# Constants
TARGET_WIDTH = 128      # Target width for resized images (larger RGB chosen size)
TARGET_HEIGHT = 64      # Target height
MIN_IMAGES_PER_CLASS = 1000

# Paths (example)
RGB_DATA_TRAIN = r"D:\eye_dataset\eye reaction last\train"
RGB_DATA_TEST = r"D:\eye_dataset\eye reaction last\test"
GRAYSCALE_DATA = r"D:\eye_dataset\Eye_cropped_6emotions"
OUTPUT_DIR = r"D:\eye_dataset\eye_dataset_1000_6classes"

CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise']


def load_images_from_folder(folder, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    images = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
         # Convert to grayscale
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_resized = cv2.resize(img_gray, size)
        images.append(img_resized)
    return images


def load_dataset_images(base_path, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    dataset_images = {}
    for cls in CLASS_NAMES:
        cls_folder = os.path.join(base_path, cls)
        if not os.path.exists(cls_folder):
            dataset_images[cls] = []
            print(f"Warning: {cls_folder} does not exist")
            continue
        imgs = load_images_from_folder(cls_folder, size)
        dataset_images[cls] = imgs
    return dataset_images


def load_grayscale_images(base_path, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    grayscale_images = {}
    for cls in CLASS_NAMES:
        cls_folder = os.path.join(base_path, cls)
        if not os.path.exists(cls_folder):
            grayscale_images[cls] = []
            print(f"Warning: {cls_folder} does not exist")
            continue
        imgs = []
        for filename in os.listdir(cls_folder):
            img_path = os.path.join(cls_folder, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img_resized = cv2.resize(img, size)
            # Convert grayscale to 3-channel by repeating channels
            img_3c = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2BGR)
            imgs.append(img_3c)
        grayscale_images[cls] = imgs
    return grayscale_images


def combine_datasets(rgb_train_path, rgb_test_path, grayscale_path, output_path):
    # Load RGB images from train and test combined
    rgb_train_images = load_dataset_images(rgb_train_path)
    rgb_test_images = load_dataset_images(rgb_test_path)
    grayscale_images = load_grayscale_images(grayscale_path)
    
    os.makedirs(output_path, exist_ok=True)
    
    for cls in CLASS_NAMES:
        rgb_imgs = rgb_train_images.get(cls, []) + rgb_test_images.get(cls, [])
        num_rgb = len(rgb_imgs)
        num_gray_needed = max(0, MIN_IMAGES_PER_CLASS - num_rgb)
        
        gray_imgs = grayscale_images.get(cls, [])
        random.shuffle(gray_imgs)  # randomize grayscale selection
        
        selected_gray = gray_imgs[:num_gray_needed]
        
        combined_imgs = rgb_imgs + selected_gray
        print(f"Class {cls}: RGB={num_rgb}, Grayscale added={len(selected_gray)}, Total={len(combined_imgs)}")
        
        # Save combined images
        class_output_folder = os.path.join(output_path, cls)
        os.makedirs(class_output_folder, exist_ok=True)
        
        for idx, img in enumerate(combined_imgs):
            save_path = os.path.join(class_output_folder, f"{cls}_{idx:03d}.png")
            cv2.imwrite(save_path, img)
            

# Example Usage

combine_datasets(
    rgb_train_path=RGB_DATA_TRAIN,
    rgb_test_path=RGB_DATA_TEST,
    grayscale_path=GRAYSCALE_DATA,
    output_path=OUTPUT_DIR
)


Class angry: RGB=72, Grayscale added=928, Total=1000
Class disgust: RGB=60, Grayscale added=940, Total=1000
Class fear: RGB=68, Grayscale added=932, Total=1000
Class happy: RGB=79, Grayscale added=921, Total=1000
Class sad: RGB=66, Grayscale added=934, Total=1000
Class surprise: RGB=63, Grayscale added=937, Total=1000
